# BERTurk Fine-tuning — Türkçe Ürün Yorumu Duygu Analizi (Hafta 2)

`dbmdz/bert-base-turkish-cased` modelini Türkçe ürün yorumu veri setinde
fine-tune eder ve **Hafta 1 baseline'ı (TF-IDF + LogReg, macro-F1 = 0.738)** ile
**dürüst şekilde** karşılaştırır.

**Dürüst kıyas için:** bu notebook repoyu klonlayıp aynı `src.data_loader` ve
aynı stratified %80/%20 bölmeyi (`random_state=42`) kullanır — yani BERTurk ile
baseline **tam olarak aynı** eğitim/test verisini görür.

## Kullanım
1. **Runtime → Change runtime type → GPU** (T4 yeterli) seç.
2. **Runtime → Run all**.
3. Sonuçlar en alttaki karşılaştırma tablosunda + confusion matrix'te.

> İlk deneme için hızlı çalıştırmak istersen aşağıdaki **Ayarlar** hücresinde
> `MAX_TRAIN_SAMPLES = 60000` yap. Tam veri (varsayılan) daha iyi sonuç verir
> ama T4'te ~1-2 saat sürebilir.

## 1. GPU kontrolü

In [ ]:
!nvidia-smi

## 2. Bağımlılıklar
Colab'da `torch` zaten var; sadece NLP paketlerini kuruyoruz.

In [ ]:
!pip install -q -U transformers datasets evaluate scikit-learn accelerate

## 3. Repoyu klonla (aynı veri + aynı bölme için)

In [ ]:
import os
REPO_URL = "https://github.com/kucukenes17/turkce-urun-yorumu-duygu-analizi.git"
if not os.path.exists('turkce-urun-yorumu-duygu-analizi'):
    !git clone -q $REPO_URL
%cd turkce-urun-yorumu-duygu-analizi
import sys; sys.path.insert(0, '.')

## 4. Ayarlar
Hepsi tek yerde; baseline ile örtüşmesi gereken değerler `src.config`'ten gelir.

In [ ]:
from src import config

MODEL_NAME = 'dbmdz/bert-base-turkish-cased'
MAX_LEN = 128            # yorumların medyanı ~19 kelime; 128 fazlasıyla yeter
BATCH_SIZE = 32
EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# None = tüm eğitim verisi (varsayılan, en iyi sonuç).
# Hızlı deneme için ör. 60000 yap (stratified alt-örnek).
MAX_TRAIN_SAMPLES = None

# Baseline ile birebir aynı bölme:
TEST_SIZE = config.TEST_SIZE          # 0.2
RANDOM_STATE = config.RANDOM_STATE    # 42
print('Baseline macro-F1 hedefi: 0.7379')

## 5. Veriyi yükle ve baseline ile AYNI şekilde böl

In [ ]:
from src.data_loader import load_reviews
from sklearn.model_selection import train_test_split

df = load_reviews()
df = df.dropna(subset=[config.TEXT_COL])
df = df[df[config.TEXT_COL].str.strip() != '']
print(f'Toplam: {len(df):,}')

train_df, test_df = train_test_split(
    df, test_size=TEST_SIZE, stratify=df[config.LABEL_COL],
    random_state=RANDOM_STATE,
)

# İsteğe bağlı hızlı örnekleme (yalnızca eğitim; test hep tam kalır → adil kıyas)
if MAX_TRAIN_SAMPLES is not None and MAX_TRAIN_SAMPLES < len(train_df):
    train_df, _ = train_test_split(
        train_df, train_size=MAX_TRAIN_SAMPLES,
        stratify=train_df[config.LABEL_COL], random_state=RANDOM_STATE)
    print(f'[not] Eğitim {MAX_TRAIN_SAMPLES:,} örneğe indirildi (hızlı deneme).')

print(f'Eğitim: {len(train_df):,} | Test: {len(test_df):,}')
print(train_df[config.LABEL_NAME_COL].value_counts())

## 6. Tokenizasyon

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_ds(frame):
    ds = Dataset.from_pandas(frame[[config.TEXT_COL, config.LABEL_COL]]                             .rename(columns={config.LABEL_COL: 'labels'}),
                             preserve_index=False)
    return ds

def tokenize(batch):
    return tokenizer(batch[config.TEXT_COL], truncation=True, max_length=MAX_LEN)

train_ds = to_ds(train_df).map(tokenize, batched=True, remove_columns=[config.TEXT_COL])
test_ds = to_ds(test_df).map(tokenize, batched=True, remove_columns=[config.TEXT_COL])
train_ds

## 7. Sınıf ağırlıkları (dengesizlik telafisi)
Baseline `class_weight='balanced'` kullanmıştı; burada da eşdeğerini
weighted cross-entropy ile uyguluyoruz.

In [ ]:
import numpy as np, torch
from sklearn.utils.class_weight import compute_class_weight

classes = np.array(sorted(config.LABEL_NAMES))  # [0, 1]
weights = compute_class_weight('balanced', classes=classes,
                               y=train_df[config.LABEL_COL].values)
class_weights = torch.tensor(weights, dtype=torch.float)
print(dict(zip([config.LABEL_NAMES[c] for c in classes], weights.round(3))))

## 8. Model, metrikler ve weighted Trainer

In [ ]:
import evaluate
from transformers import (AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, DataCollatorWithPadding)
from sklearn.metrics import f1_score

id2label = {int(k): v for k, v in config.LABEL_NAMES.items()}
label2id = {v: int(k) for k, v in config.LABEL_NAMES.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
acc_metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1_per = f1_score(labels, preds, average=None, labels=[0, 1])
    return {
        'accuracy': acc_metric.compute(predictions=preds, references=labels)['accuracy'],
        'f1_macro': f1_score(labels, preds, average='macro'),
        'f1_negatif': f1_per[0],
        'f1_pozitif': f1_per[1],
    }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(outputs.logits.device))
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

## 9. Eğitim

In [ ]:
args = TrainingArguments(
    output_dir='berturk-sentiment',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    report_to='none',
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=test_ds,
    tokenizer=tokenizer, data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## 10. Değerlendirme + baseline ile karşılaştırma

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

pred = trainer.predict(test_ds)
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=-1)
names = [config.LABEL_NAMES[i] for i in sorted(config.LABEL_NAMES)]

print(classification_report(y_true, y_pred, target_names=names, digits=4))

cm = confusion_matrix(y_true, y_pred, labels=sorted(config.LABEL_NAMES))
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=names).plot(ax=ax, cmap='Blues', colorbar=False, values_format=',')
ax.set_title('BERTurk — Confusion Matrix'); plt.tight_layout(); plt.show()

In [ ]:
import pandas as pd
berturk_f1 = f1_score(y_true, y_pred, average='macro')
cmp = pd.DataFrame({
    'model': ['Baseline (TF-IDF + LogReg)', 'BERTurk (fine-tuned)'],
    'macro_f1': [0.7379, round(berturk_f1, 4)],
})
print(cmp.to_string(index=False))
delta = berturk_f1 - 0.7379
print(f"\nBERTurk baseline'i {'GECTI' if delta > 0 else 'GECEMEDI'} "
      f"(fark: {delta:+.4f})")

## 11. Modeli kaydet (Colab)
İstersen Google Drive'a veya HF Hub'a da yükleyebilirsin.

In [ ]:
trainer.save_model('berturk-sentiment-final')
tokenizer.save_pretrained('berturk-sentiment-final')
print('Model kaydedildi: berturk-sentiment-final/')
# Drive'a kaydetmek için:
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r berturk-sentiment-final /content/drive/MyDrive/